# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chạy từ trên xuống. Runtime > Change runtime type > **T4 GPU** trước khi bắt đầu.

| Ô | Làm gì | Thời gian |
|---|---|---|
| 1 | clone repo nộp bài + install | ~1 phút |
| 2 | smoke: import + unit test | ~30 giây |
| 3 | **core pipeline NB1 → NB5** | ~80 phút |
| 4 | gatekeeper + in kết quả | ~10 giây |

> **Bản nộp bài để `EVAL_LIMIT = ""`.** Ô 3 mặc định chạy **toàn bộ** tập eval.
> Chỉ hạ xuống `"8"` khi cần smoke/debug — lúc đó `results/baselines_frozen.json` ghi
> `smoke_mode=true` và `scripts/verify.py` báo FAIL, tức là không đủ điều kiện nộp.


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

# Repo NỘP BÀI của sinh viên (không phải upstream của giảng viên).
REPO = "https://github.com/TaVanTuan24/Day21-Track3-TaVanTuan-2A202602806-Finetuning-Lab.git"
# Clone vào một WORKDIR tường minh. KHÔNG suy ra tên thư mục từ tên repo upstream —
# thư mục clone và os.chdir() phải khớp nhau bất kể fork tên gì.
WORKDIR = "/content/Day21-Track3-Finetuning-Lab"

if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    if os.path.exists(WORKDIR):
        raise RuntimeError(
            f"{WORKDIR} đã tồn tại nhưng không phải git clone. Xoá nó rồi chạy lại ô này."
        )
    subprocess.run(["git", "clone", "-q", REPO, WORKDIR], check=True)
os.chdir(WORKDIR)
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("repo   :", REPO)
print("workdir:", os.getcwd())
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


## Committed evidence configuration — đọc trước khi chạy

| Ô 3 ở dưới mặc định chạy đường **T4** của lab (model 4B) | |
|---|---|
| `COMPUTE_TIER` | `T4` — đường chính thức của lab, cần Colab T4 16 GB |
| `EVAL_LIMIT` | `""` — để trống, đây là bản nộp bài |

Nhưng `results/` trong repo này **không** được tạo bằng cấu hình mặc định đó. Evidence đã
commit được tạo bằng:

```
COMPUTE_TIER=LAPTOP
BASE_MODEL=Qwen/Qwen3.5-0.8B
MASK_MODE=assistant-only
EPOCHS=2
# EVAL_LIMIT không đặt
```

Lý do: máy huấn luyện là laptop GPU **4 GB VRAM**, không nạp nổi model 4B của tier T4. Đây
là cấu hình đầy đủ, dạng máy đọc được: `submission/RUN_CONFIG.env`. Giải thích: `REPORT.md §1`.

**Muốn tái lập đúng các số đã commit?** Đặt `COMPUTE_TIER = "LAPTOP"` và
`BASE_MODEL = "Qwen/Qwen3.5-0.8B"` trong ô 3 (hoặc copy ô dưới). Chạy mặc định thì bạn sẽ
tái lập **đường T4 của lab**, và các con số sẽ **khác** `results/` — đó là dự kiến, không
phải lỗi.


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# ĐÂY LÀ ĐƯỜNG T4 CỦA LAB (model 4B). results/ đã commit được tạo bằng LAPTOP +
# Qwen/Qwen3.5-0.8B, không phải bằng mặc định này — xem ô markdown ngay trên, hoặc
# submission/RUN_CONFIG.env, hoặc chạy ô "3b. Tái lập committed evidence".
# EVAL_LIMIT truncates both eval sets. "" = full run (BẢN NỘP BÀI — để trống).
# "8" chỉ dùng để smoke/debug cho nhanh: results/ sẽ ghi lại smoke_mode=True và
# scripts/verify.py sẽ FAIL "full eval set used". STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
BASE_MODEL   = ""          # @param {type:"string"}   # trống = model mặc định của tier
EVAL_LIMIT   = ""          # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if BASE_MODEL:
    os.environ["BASE_MODEL"] = BASE_MODEL
else:
    os.environ.pop("BASE_MODEL", None)
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 3b. Tái lập committed evidence (tùy chọn)
# Cấu hình đã tạo ra results/ trong repo này. Bỏ qua ô này nếu bạn muốn chạy đường T4.
import os, subprocess, sys

os.environ["COMPUTE_TIER"] = "LAPTOP"
os.environ["BASE_MODEL"] = "Qwen/Qwen3.5-0.8B"
os.environ["MASK_MODE"] = "assistant-only"
os.environ["EPOCHS"] = "2"
os.environ.pop("EVAL_LIMIT", None)          # bản nộp bài: chạy TOÀN BỘ tập eval

from labkit import config, device
_t = config.get_tier()
print(device.banner(), "\n")
print(f"tier={_t.name}  model={_t.model_id}  max_length={_t.max_length}  "
      f"effective_batch={_t.effective_batch}  epochs={os.environ['EPOCHS']}  "
      f"eval_limit={'FULL' if 'EVAL_LIMIT' not in os.environ else os.environ['EVAL_LIMIT']}")

subprocess.run([sys.executable, "-u", "scripts/colab_run.py", "nb1", "nb2", "nb3", "nb4", "nb5"],
               check=True)


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null
